In [1]:
!pip install transformers datasets accelerate -q
!pip install --upgrade transformers datasets accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 39.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


In [2]:
import torch
print("CUDA Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0))

CUDA Available: True
GPU Name: Tesla T4


In [3]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/testesterone/medquad-cleaned/medquad_cleaned.csv")
print(df.shape)
df.head()

(16359, 2)


,question,answer
0,what is are glaucoma ?,glaucoma is a group of diseases that can damag...
1,what causes glaucoma ?,"nearly 2.7 million people have glaucoma, a lea..."
2,what are the symptoms of glaucoma ?,symptoms of glaucoma glaucoma can develop in o...
3,what are the treatments for glaucoma ?,"although openangle glaucoma cannot be cured, i..."
4,what is are glaucoma ?,glaucoma is a group of diseases that can damag...


In [4]:
from sklearn.model_selection import train_test_split

# First split = training (80%) and temporary (20%)
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

# Second split = validation (10%) and test (10%)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42
)

print("Training set:", train_df.shape)
print("Validation set:", val_df.shape)
print("Test set:", test_df.shape)


Training set: (13087, 2)
Validation set: (1636, 2)
Test set: (1636, 2)


In [5]:
# converting from pandas dataframe to huggingface dataset
from datasets import Dataset
train_dataset = Dataset.from_pandas(train_df)
val_dataset   = Dataset.from_pandas(val_df)
test_dataset  = Dataset.from_pandas(test_df)
train_dataset = train_dataset.remove_columns(["__index_level_0__"])
val_dataset   = val_dataset.remove_columns(["__index_level_0__"])
test_dataset  = test_dataset.remove_columns(["__index_level_0__"])
train_dataset[0]



{'question': 'how many people are affected by cranioectodermal dysplasia ?',
 'answer': 'cranioectodermal dysplasia is a rare condition with an unknown prevalence. approximately 40 cases of this condition have been described in the medical literature. this information is for general guidance only and does not replace professional medical advice.'}

In [6]:
from transformers import AutoTokenizer
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)



config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [7]:
def tokenize_function(batch):
    # Tokenize input 
    model_inputs = tokenizer(
        batch["question"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

    # Tokenize output 
    labels = tokenizer(
        text_target=batch["answer"],
        padding="max_length",
        truncation=True,
        max_length=200
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


In [8]:
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset   = val_dataset.map(tokenize_function, batched=True)


Map:   0%|          | 0/13087 [00:00<?, ? examples/s]

Map:   0%|          | 0/1636 [00:00<?, ? examples/s]

In [9]:
train_dataset = train_dataset.remove_columns(
    ["question", "answer"]
)

val_dataset = val_dataset.remove_columns(
    ["question", "answer"]
)

In [10]:
import torch
print(torch.__version__)

2.9.0+cu126


In [11]:
from transformers import AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [12]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    save_total_limit=1,
    report_to="none",
    fp16=True,
    logging_steps=10
)

NameError: name 'TrainingArguments' is not defined

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model("/kaggle/working/medical-chatbot-model")
tokenizer.save_pretrained("/kaggle/working/medical-chatbot-model")

In [ ]:
!zip -r medical-chatbot-model.zip /kaggle/working/medical-chatbot-model